In [1]:
import os
import pandas as pd
import numpy as np
import trimesh
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [4]:
excel_path = r"D:/Professional/Projects/Aiims/Gait/Data/project work/PROJECTBook1.xlsx"
df_cobb = pd.read_excel(excel_path)

# Drop where we don't have cobb angle

df_cobb_s = df_cobb[~df_cobb["COBB'S"].isna()]

# Assuming 'PATIENT_NAME' matches folder name, 'COBB\'S' has Cobb angle
patient_to_cobb = dict(zip(df_cobb_s['NAME'], df_cobb_s["COBB'S"]))


In [59]:
patient_to_cobb

{'Nishika': 20.0,
 'Ishika Goher': 41.0,
 'Humera Bi': 33.2,
 'Tapsya': 29.0,
 'Sudhanshu Singh': 64.0,
 'Varsha Mishra': 69.0,
 'Amit Sharma': 49.0,
 'Aditya Gahrola': 65.0,
 'Nidhi': 60.0,
 'Palak': 37.5,
 'Mrinmoy Saha': 71.0,
 'Maya ': 41.0,
 'Satyam pal': 36.0,
 'Aahil': 78.0,
 'Nakul': 52.0,
 'Smriti Nitya': 34.0,
 'Aafiya Saifi': 56.0,
 'Aarav Tiwari': 44.0,
 'Aradhya Sahu': 40.0,
 'Divyanka': 61.0,
 'Surjeet Barman': 38.0,
 'Karan Bhati': 57.0,
 'Khushi Gupta': 52.0,
 'Gracy Singh': 44.0,
 'Ayan Saifi': 30.0,
 'karan kumar': 72.0,
 'Radhika Sharma': 67.0,
 'Sayba Bano': 36.0,
 'Imran ALI': 64.0,
 'Talib Khan': 111.0,
 'Gargi jha': 60.0,
 'Bhawna sheoran(o)': 54.0,
 'Muskaan khatun': 39.0,
 'Hitesh Sharma': 68.0,
 'Kartik Verma': 71.0,
 'Neha .delhi': 32.0,
 'Anamika Sourout': 35.0,
 'baishakhi debnath(o)': 58.0}

In [6]:
len(patient_to_cobb)

38

### Exploration

In [7]:
import open3d as o3d
import os
import numpy as np

def crop_torso(stl_path, 
               x_range=None, y_range=None, z_range=None,
               z_lower_frac=0.2, z_upper_frac=0.8,
               xy_margin_frac=0.05):
    """
    Crops torso region from STL mesh, either automatically or using manual bounds.
    
    Args:
        stl_path (str): Input STL file path.
        x_range, y_range, z_range (tuple or None): Manual bounding box in mm.
        z_lower_frac, z_upper_frac (float): Fraction of height to keep if z_range is None.
        xy_margin_frac (float): Fraction to trim from sides if x/y ranges are None.
    
    Returns:
        o3d.geometry.TriangleMesh: Cropped mesh.
        str: Saved STL path.
    """
    # Load mesh
    mesh = o3d.io.read_triangle_mesh(stl_path)
    if mesh.is_empty():
        raise ValueError("Mesh is empty!")
    
    # Compute bounds for auto mode
    min_bound = mesh.get_min_bound()
    max_bound = mesh.get_max_bound()
    dims = max_bound - min_bound

    # Determine ranges
    if x_range is None:
        x_range = (min_bound[0] + xy_margin_frac*dims[0], max_bound[0] - xy_margin_frac*dims[0])
    if y_range is None:
        y_range = (min_bound[1] + xy_margin_frac*dims[1], max_bound[1] - xy_margin_frac*dims[1])
    if z_range is None:
        z_range = (min_bound[2] + z_lower_frac*dims[2], min_bound[2] + z_upper_frac*dims[2])

    # Crop bounding box
    bbox = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=(x_range[0], y_range[0], z_range[0]),
        max_bound=(x_range[1], y_range[1], z_range[1])
    )
    
    torso_mesh = mesh.crop(bbox)
    if torso_mesh.is_empty():
        raise ValueError("Cropped mesh is empty! Adjust bounds.")

    # Compute normals for cropped mesh
    torso_mesh.compute_vertex_normals()

    # Clean mesh
    torso_mesh.remove_duplicated_vertices()
    torso_mesh.remove_duplicated_triangles()
    torso_mesh.remove_non_manifold_edges()
    torso_mesh.remove_degenerate_triangles()

    # Save automatically
    folder = os.path.join(os.path.dirname(stl_path), "cropped_meshes")
    os.makedirs(folder, exist_ok=True)
    filename = os.path.basename(stl_path)
    save_path = os.path.join(folder, f"cropped_{filename}")
    o3d.io.write_triangle_mesh(save_path, torso_mesh)
    
    print(f"Cropped torso saved at: {save_path}")
    return torso_mesh, save_path


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [52]:
import open3d as o3d

mesh = o3d.io.read_triangle_mesh(
    "D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi/150_1.stl"
)
print("Mesh min bound:", mesh.get_min_bound())
print("Mesh max bound:", mesh.get_max_bound())
print("Mesh dimensions:", mesh.get_max_bound() - mesh.get_min_bound())


Mesh min bound: [-0.74804688 -0.74804688 -1.99804688]
Mesh max bound: [ 0.74804688  0.74804688 -0.79790902]
Mesh dimensions: [1.49609375 1.49609375 1.20013785]


In [47]:
torso_mesh, saved_path = crop_torso(
    "D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi/140_1.stl",
    x_range=(-0.2, 0.5),
    y_range=(-0.6, 0.698),
    z_range=(-1.5, -1.13085938)
)

print("Cropped mesh saved at:", saved_path)
o3d.visualization.draw_geometries([torso_mesh])


Cropped torso saved at: D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi\cropped_meshes\cropped_140_1.stl
Cropped mesh saved at: D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi\cropped_meshes\cropped_140_1.stl


In [ ]:
torso_mesh, saved_path = crop_torso(
    "D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi/120_1.stl",
    x_range=(-0.6, 0.5),
    y_range=(-0.6, 0.73),
    z_range=(-1.6, -1.01757812)
)

print("Cropped mesh saved at:", saved_path)
o3d.visualization.draw_geometries([torso_mesh])


Cropped torso saved at: D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi\cropped_meshes\cropped_120_1.stl
Cropped mesh saved at: D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi\cropped_meshes\cropped_120_1.stl


### Fn to crop 3d file

In [48]:
import open3d as o3d
import numpy as np
import os

def crop_torso_freeze_xy_dynamic_z(
    stl_path,
    x_range=(-0.6, 0.5),
    y_range=(-0.6, 0.73),
    z_frac=(0.85, 1.0),   # keep last 15% (top region)
    visualize_bbox=False
):
    """
    Crops a mesh using fixed X/Y bounds and dynamic Z fraction.

    Args:
        stl_path (str): Path to input STL mesh.
        x_range, y_range (tuple): Fixed crop bounds for X and Y axes.
        z_frac (tuple): Fraction of Z range to keep (e.g., (0.85, 1.0) keeps top 15%).
        visualize_bbox (bool): If True, show the mesh with the crop bounding box.

    Returns:
        torso_mesh (o3d.geometry.TriangleMesh): Cropped mesh.
        save_path (str): Path where cropped STL is saved.
        used_bounds (dict): Dictionary of bounds used.
    """
    # Load mesh
    mesh = o3d.io.read_triangle_mesh(stl_path)
    if mesh.is_empty():
        raise ValueError("❌ Mesh is empty or failed to load.")

    # Get mesh bounds
    min_bound = np.array(mesh.get_min_bound())
    max_bound = np.array(mesh.get_max_bound())
    dims = max_bound - min_bound

    # Compute Z range based on fractions
    z_min = min_bound[2] + z_frac[0] * dims[2]
    z_max = min_bound[2] + z_frac[1] * dims[2]

    # Build crop bounding box
    bbox = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=(x_range[0], y_range[0], z_min),
        max_bound=(x_range[1], y_range[1], z_max)
    )
    bbox.color = (1, 0, 0)  # red for visualization

    if visualize_bbox:
        o3d.visualization.draw_geometries([mesh, bbox], window_name="Original Mesh + Crop Box")

    # Crop
    torso_mesh = mesh.crop(bbox)
    if torso_mesh.is_empty():
        raise ValueError("❌ Cropped mesh is empty. Adjust Z fractions or XY ranges.")

    # Clean mesh
    torso_mesh.remove_duplicated_vertices()
    torso_mesh.remove_duplicated_triangles()
    torso_mesh.remove_degenerate_triangles()
    try:
        torso_mesh.remove_non_manifold_edges()
    except Exception:
        pass

    torso_mesh.compute_vertex_normals()
    torso_mesh.compute_triangle_normals()

    # Save cropped mesh
    folder = os.path.join(os.path.dirname(stl_path), "cropped_meshes")
    os.makedirs(folder, exist_ok=True)
    filename = os.path.basename(stl_path)
    save_path = os.path.join(folder, f"cropped_{filename}")
    o3d.io.write_triangle_mesh(save_path, torso_mesh)

    used_bounds = {
        "x_range": x_range,
        "y_range": y_range,
        "z_range": (z_min, z_max),
        "mesh_min": min_bound.tolist(),
        "mesh_max": max_bound.tolist(),
    }

    print(f"✅ Cropped torso saved at: {save_path}")
    print(f"📏 Used bounds: {used_bounds}")

    if visualize_bbox:
        o3d.visualization.draw_geometries([torso_mesh], window_name="Cropped Mesh")

    return torso_mesh, save_path, used_bounds


In [ ]:
torso_mesh, save_path, bounds = crop_torso_freeze_xy_dynamic_z(
    r"D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi/100_1.stl",
    x_range=(-0.5, 0.5),
    y_range=(-0.6, 0.73),
    z_frac=(0.15, 1.0),   
    visualize_bbox=False
)
print("Saved at:", save_path)
o3d.visualization.draw_geometries([torso_mesh])


✅ Cropped torso saved at: D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi\cropped_meshes\cropped_100_1.stl
📏 Used bounds: {'x_range': (-0.5, 0.5), 'y_range': (-0.6, 0.73), 'z_range': (np.float64(-1.7982421875), np.float64(-0.666015625)), 'mesh_min': [-0.748046875, -0.748046875, -1.998046875], 'mesh_max': [0.748046875, 0.748046875, -0.666015625]}
Saved at: D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi\cropped_meshes\cropped_100_1.stl


In [58]:
import os
import re
import glob
import csv

# === CONFIG ===
BASE_PATIENTS_DIR = r"D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All patients"
OUTPUT_DIR = r"D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped"
LOG_CSV = os.path.join(OUTPUT_DIR, "cropping_log.csv")
# XY/Z settings (if you want to override defaults passed to the function)
X_RANGE = (-0.6, 0.5)
Y_RANGE = (-0.6, 0.73)
Z_FRAC = (0.15, 1.0)

# Behavior
SKIP_IF_EXISTS = True
OVERWRITE = False  # if True, it will overwrite existing cropped files
WRITE_LOG = True

# Regex to match files like "100_1.stl" or "150_2.stl" (case-insensitive)
STL_FILENAME_RE = re.compile(r'^(?P<dist>\d{2,3})_(?P<scan>\d+)\.stl$', flags=re.IGNORECASE)


# === helper functions ===
def patient_shortname_from_folder(folder_name):
    """
    Convert folder name like '1_Ayan_Saifi' or 'Ayan_Saifi' to 'ayansaifi' (lowercase, no non-alnum).
    If the folder starts with a number + underscore, we drop the leading index.
    """
    name = folder_name
    if "_" in folder_name:
        # drop leading token if it is purely digits (e.g., "1_Ayan_Saifi")
        first, rest = folder_name.split("_", 1)
        if first.isdigit():
            name = rest
    # Normalize: lowercase and remove non-alphanumeric characters
    name = re.sub(r'[^0-9a-zA-Z]', '', name).lower()
    return name


def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p


# === main batch function ===
def batch_crop_all_patients(
    base_dir=BASE_PATIENTS_DIR,
    out_dir=OUTPUT_DIR,
    x_range=X_RANGE,
    y_range=Y_RANGE,
    z_frac=Z_FRAC,
    skip_if_exists=SKIP_IF_EXISTS,
    overwrite=OVERWRITE,
    write_log=WRITE_LOG,
    log_csv_path=LOG_CSV
):
    ensure_dir(out_dir)
    rows = []
    patient_folders = [d for d in glob.glob(os.path.join(base_dir, "*")) if os.path.isdir(d)]

    if not patient_folders:
        print("No patient folders found in", base_dir)
        return

    for pf in sorted(patient_folders):
        folder_name = os.path.basename(pf)
        patient_short = patient_shortname_from_folder(folder_name)

        # find STL files in that patient folder
        stl_files = sorted(glob.glob(os.path.join(pf, "*.stl")))
        if not stl_files:
            print(f"Skipping {folder_name}: no .stl files found.")
            continue

        for stl in stl_files:
            stl_fname = os.path.basename(stl)
            m = STL_FILENAME_RE.match(stl_fname)
            if not m:
                print(f"Skipping file (unexpected name format): {stl_fname}")
                continue

            dist = m.group("dist")
            scan_idx = m.group("scan")
            out_basename = f"{dist}_{scan_idx}_{patient_short}_cropped.stl"
            out_path = os.path.join(out_dir, out_basename)

            if os.path.exists(out_path):
                if skip_if_exists and not overwrite:
                    print(f"Skipping (exists): {out_basename}")
                    continue
                elif overwrite:
                    print(f"Overwriting: {out_basename}")

            try:
                # Call your crop function (make sure it's in scope)
                torso_mesh, saved_path, used_bounds = crop_torso_freeze_xy_dynamic_z(
                    stl,
                    x_range=x_range,
                    y_range=y_range,
                    z_frac=z_frac,
                    visualize_bbox=False
                )
                # crop_torso_freeze_xy_dynamic_z saves to its own folder; we want to move (or re-save) to single output folder
                # If the function already returned saved_path, read and rewrite to our desired filename to unify output location.
                # We will attempt to re-save the mesh using open3d to the output location.
                import open3d as o3d
                o3d.io.write_triangle_mesh(out_path, torso_mesh)
                print(f"Saved: {out_basename}")

                log_row = {
                    "patient_folder": folder_name,
                    "patient_id": patient_short,
                    "input_file": stl,
                    "output_file": out_path,
                    "dist": dist,
                    "scan_idx": scan_idx,
                    "used_x_min": used_bounds["x_range"][0],
                    "used_x_max": used_bounds["x_range"][1],
                    "used_y_min": used_bounds["y_range"][0],
                    "used_y_max": used_bounds["y_range"][1],
                    "used_z_min": used_bounds["z_range"][0],
                    "used_z_max": used_bounds["z_range"][1],
                }
                rows.append(log_row)

            except Exception as e:
                print(f"Error processing {stl_fname}: {e}")
                rows.append({
                    "patient_folder": folder_name,
                    "patient_id": patient_short,
                    "input_file": stl,
                    "output_file": None,
                    "dist": dist if 'dist' in locals() else None,
                    "scan_idx": scan_idx if 'scan_idx' in locals() else None,
                    "used_x_min": None,
                    "used_x_max": None,
                    "used_y_min": None,
                    "used_y_max": None,
                    "used_z_min": None,
                    "used_z_max": None,
                    "error": str(e)
                })

    # write csv log if requested
    if write_log and rows:
        fieldnames = [
            "patient_folder", "patient_id", "input_file", "output_file",
            "dist", "scan_idx",
            "used_x_min", "used_x_max", "used_y_min", "used_y_max", "used_z_min", "used_z_max",
            "error"
        ]
        # ensure output dir exists
        ensure_dir(os.path.dirname(log_csv_path))
        with open(log_csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for r in rows:
                # ensure all keys exist
                r_complete = {k: r.get(k, None) for k in fieldnames}
                writer.writerow(r_complete)
        print(f"\nLog written to: {log_csv_path}")

    print("\nBatch cropping complete.")


# === run ===
if __name__ == "__main__":
    batch_crop_all_patients(
        base_dir=BASE_PATIENTS_DIR,
        out_dir=OUTPUT_DIR,
        x_range=X_RANGE,
        y_range=Y_RANGE,
        z_frac=Z_FRAC,
        skip_if_exists=SKIP_IF_EXISTS,
        overwrite=OVERWRITE,
        write_log=WRITE_LOG,
        log_csv_path=LOG_CSV
    )


✅ Cropped torso saved at: D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All patients\1_Ayan_Saifi\cropped_meshes\cropped_100_1.stl
📏 Used bounds: {'x_range': (-0.6, 0.5), 'y_range': (-0.6, 0.73), 'z_range': (np.float64(-1.8132466793060302), np.float64(-0.7660455703735352)), 'mesh_min': [-0.748046875, -0.748046875, -1.998046875], 'mesh_max': [0.748046875, 0.748046875, -0.7660455703735352]}
Saved: 100_1_ayansaifi_cropped.stl
✅ Cropped torso saved at: D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All patients\1_Ayan_Saifi\cropped_meshes\cropped_100_2.stl
📏 Used bounds: {'x_range': (-0.6, 0.5), 'y_range': (-0.6, 0.73), 'z_range': (np.float64(-1.8143573373556137), np.float64(-0.7734499573707581)), 'mesh_min': [-0.748046875, -0.748046875, -1.998046875], 'mesh_max': [0.748046875, 0.748046875, -0.7734499573707581]}
Saved: 100_2_ayansaifi_cropped.stl
✅ Cropped torso saved at: D:\Professional\Projects\Aiims\Gait\Data\project work\Su

Fn to extract features from 3d files

In [62]:
import os
import glob
import re
import csv
import json
import numpy as np

import open3d as o3d
# optional: trimesh for better volume calculation
try:
    import trimesh
except ImportError:
    trimesh = None

# ---------- CONFIG ----------
ALL_CROPPED_DIR = r"D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped"
OUT_CSV = os.path.join(ALL_CROPPED_DIR, "all_cropped_features.csv")
OUT_JSON = os.path.join(ALL_CROPPED_DIR, "all_cropped_features.json")

# slice percentiles (you can change)
SLICE_PERCENTILES = [0.95, 0.90, 0.85, 0.80, 0.75]

# filename regex: dist_scan_patient_cropped(.stl)
FNAME_RE = re.compile(r'^(?P<dist>\d{2,3})_(?P<scan>\d+)_(?P<patient>.+?)_cropped', flags=re.IGNORECASE)

# ---------- helper functions ----------
def parse_fname(fname):
    """Return (distance:int or None, scan:int or None, patient:str or filename_without_ext)"""
    bn = os.path.basename(fname)
    name_no_ext = os.path.splitext(bn)[0]
    m = FNAME_RE.match(name_no_ext)
    if m:
        return int(m.group("dist")), int(m.group("scan")), m.group("patient")
    # fallback: try splitting by underscores and pick token 2 as patient if length>=3
    parts = name_no_ext.split("_")
    if len(parts) >= 3:
        try:
            dist = int(parts[0])
            scan = int(parts[1])
            patient = "_".join(parts[2:-1]) if parts[-1].lower().startswith("cropped") else "_".join(parts[2:])
            return dist, scan, patient
        except Exception:
            pass
    # final fallback
    return None, None, name_no_ext

def circumference_of_slice(pts2d):
    """Perimeter of convex hull for 2D points (simple fallback implemented)."""
    if pts2d.shape[0] < 3:
        return 0.0
    # try scipy ConvexHull if available
    try:
        from scipy.spatial import ConvexHull
        hull = ConvexHull(pts2d)
        hull_pts = pts2d[hull.vertices]
    except Exception:
        # monotone chain convex hull (Andrew) fallback
        pts = np.asarray(sorted(map(tuple, pts2d)))
        def cross(o,a,b): return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])
        lower = []
        for p in pts:
            while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
                lower.pop()
            lower.append(p)
        upper = []
        for p in reversed(pts):
            while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
                upper.pop()
            upper.append(p)
        hull_pts = np.array(lower[:-1] + upper[:-1])
    diffs = np.diff(np.vstack([hull_pts, hull_pts[0]]), axis=0)
    return float(np.sqrt((diffs**2).sum(axis=1)).sum())

def extract_features(stl_path, slice_percentiles=SLICE_PERCENTILES):
    mesh = o3d.io.read_triangle_mesh(stl_path)
    if mesh.is_empty():
        raise ValueError(f"Mesh empty or failed to load: {stl_path}")
    verts = np.asarray(mesh.vertices)
    tris = np.asarray(mesh.triangles)
    if verts.size == 0:
        raise ValueError("No vertices found.")

    # ensure normals
    if not mesh.has_vertex_normals():
        mesh.compute_vertex_normals()

    aabb = mesh.get_axis_aligned_bounding_box()
    bbox_min = aabb.min_bound
    bbox_max = aabb.max_bound
    bbox_dims = bbox_max - bbox_min

    num_vertices = int(verts.shape[0])
    num_triangles = int(tris.shape[0])
    surface_area = float(mesh.get_surface_area())
    centroid = verts.mean(axis=0).tolist()
    height = float(bbox_dims[2])

    # volume via trimesh if available (more reliable)
    volume = None
    is_watertight = None
    if trimesh is not None:
        try:
            t = trimesh.Trimesh(vertices=verts, faces=tris, process=False)
            volume = float(getattr(t, "volume", 0.0))
            is_watertight = bool(getattr(t, "is_watertight", False))
        except Exception:
            volume = None
            is_watertight = None

    # slice circumferences
    slice_info = {}
    zmin, zmax = bbox_min[2], bbox_max[2]
    for p in slice_percentiles:
        zq = zmin + p * (zmax - zmin)
        slab = max(1e-6, 0.005 * max(1e-6, height))
        mask = (verts[:,2] >= zq - slab) & (verts[:,2] <= zq + slab)
        pts2d = verts[mask][:, :2]
        circ = circumference_of_slice(pts2d) if pts2d.shape[0] >= 3 else 0.0
        slice_info[f"slice_{int(p*100)}"] = circ

    features = {
        "file": os.path.basename(stl_path),
        "num_vertices": num_vertices,
        "num_triangles": num_triangles,
        "surface_area": surface_area,
        "volume": volume,
        "is_watertight": is_watertight,
        "bbox_dx": float(bbox_dims[0]),
        "bbox_dy": float(bbox_dims[1]),
        "bbox_dz": float(bbox_dims[2]),
        "centroid_x": float(centroid[0]),
        "centroid_y": float(centroid[1]),
        "centroid_z": float(centroid[2]),
        "height": height,
        **slice_info
    }
    return features

# ---------- main ----------
def main():
    files = sorted(glob.glob(os.path.join(ALL_CROPPED_DIR, "*cropped*.stl")))
    if not files:
        print("No cropped files found in:", ALL_CROPPED_DIR)
        return

    rows = []
    for fp in files:
        try:
            dist, scan, patient = parse_fname(os.path.basename(fp))
            feats = extract_features(fp)
            # flatten and add patient & distance
            row = feats.copy()
            row["patient"] = patient
            row["distance_cm"] = dist
            row["input_path"] = fp
            rows.append(row)
            print(f"Processed: {feats['file']}  patient: {patient}  dist: {dist}")
        except Exception as e:
            print(f"Failed {fp}: {e}")
            rows.append({"file": os.path.basename(fp), "input_path": fp, "error": str(e)})

    # write CSV
    fieldnames = sorted(set().union(*(r.keys() for r in rows)))
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)
    print("CSV saved to:", OUT_CSV)

    # write JSON
    with open(OUT_JSON, "w", encoding="utf-8") as jf:
        json.dump(rows, jf, indent=2)
    print("JSON saved to:", OUT_JSON)

if __name__ == "__main__":
    main()


Processed: 100_1_anamika_cropped.stl  patient: anamika  dist: 100
Processed: 100_1_ayansaifi_cropped.stl  patient: ayansaifi  dist: 100
Processed: 100_1_bhavna_cropped.stl  patient: bhavna  dist: 100
Processed: 100_1_gargi_cropped.stl  patient: gargi  dist: 100
Processed: 100_1_gracysingh_cropped.stl  patient: gracysingh  dist: 100
Processed: 100_1_hiteshsharma_cropped.stl  patient: hiteshsharma  dist: 100
Processed: 100_1_imranali_cropped.stl  patient: imranali  dist: 100
Processed: 100_1_karan_cropped.stl  patient: karan  dist: 100
Processed: 100_1_radhikasharma_cropped.stl  patient: radhikasharma  dist: 100
Processed: 100_1_s1karan_cropped.stl  patient: s1karan  dist: 100
Processed: 100_1_s1neha_cropped.stl  patient: s1neha  dist: 100
Processed: 100_1_s2karthik_cropped.stl  patient: s2karthik  dist: 100
Processed: 100_1_s2surjeet_cropped.stl  patient: s2surjeet  dist: 100
Processed: 100_1_s3khushi_cropped.stl  patient: s3khushi  dist: 100
Processed: 100_1_s3muskhan_cropped.stl  pati

In [63]:
import re
import csv
import json
import difflib
import os

# ---------- CONFIG ----------
FEATURES_CSV = r"D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\all_cropped_features.csv"
OUTPUT_CSV = FEATURES_CSV.replace(".csv", "_with_cobb.csv")
UNMATCHED_CSV = FEATURES_CSV.replace(".csv", "_cobb_unmatched.csv")
MAPPING_LOG_JSON = FEATURES_CSV.replace(".csv", "_cobb_map_log.json")

# your cobb dictionary (example)
COBB_DICT = patient_to_cobb

# ---------- helpers ----------
def normalize_name(name):
    """
    Normalize patient name for matching:
    - strip leading/trailing whitespace
    - remove leading s1..s4 or s01..s04 tokens and any leading digits + underscores
    - remove non-alphanumeric characters
    - lowercase and collapse spaces
    - return as compact string (no spaces) for robust matching
    Examples:
      "s1_vaishakhiDevnath" -> "vaishakhidevnath"
      "Ishika Goher" -> "ishikagoher"
    """
    if name is None:
        return ""
    s = name.strip()
    # remove common noisy prefixes e.g., s1_, s2-, s3, s04_, 1_, etc.
    s = re.sub(r'^(s[1-4]|s0?[1-4]|\d+)[_\-\. ]*', '', s, flags=re.IGNORECASE)
    # remove any remaining non-alphanumeric (keep letters and digits)
    s = re.sub(r'[^0-9A-Za-z]', '', s)
    s = s.lower()
    return s

def build_normalized_dict(cobb_dict):
    """
    Return dict mapping normalized_name -> (orig_name, cobb_value)
    If two different original names normalize to same key, keep the first but log a warning.
    """
    norm_map = {}
    collisions = {}
    for orig, val in cobb_dict.items():
        n = normalize_name(orig)
        if n in norm_map:
            collisions.setdefault(n, []).append((orig, val))
        else:
            norm_map[n] = (orig, val)
    return norm_map, collisions

# ---------- main mapping function ----------
def map_cobb_to_csv(features_csv=FEATURES_CSV, out_csv=OUTPUT_CSV, unmatched_csv=UNMATCHED_CSV,
                    cobb_dict=COBB_DICT, log_json=MAPPING_LOG_JSON, fuzzy_cutoff=0.8, keep_original_patient_col=True):
    # load features CSV
    if not os.path.exists(features_csv):
        raise FileNotFoundError(f"Features CSV not found: {features_csv}")

    norm_cobb_map, collisions = build_normalized_dict(cobb_dict)
    # prepare name lists for fuzzy matching
    norm_names = list(norm_cobb_map.keys())

    mapped_rows = []
    unmatched_rows = []
    map_log = {"mapped": [], "fuzzy_mapped": [], "unmatched": [], "collisions": collisions}

    with open(features_csv, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        fieldnames = list(reader.fieldnames) if reader.fieldnames else []
        # add cobb column if missing
        if "cobb_angle" not in fieldnames:
            fieldnames.append("cobb_angle")
        # keep original patient string column name (some CSVs use 'patient' column); ensure it's in header
        patient_col_candidates = ["patient", "Patient", "patient_name", "name"]
        patient_col = None
        for cand in patient_col_candidates:
            if cand in fieldnames:
                patient_col = cand
                break
        # If no existing patient column, assume filename column 'file' encodes patient as third token
        if patient_col is None and "file" in fieldnames:
            patient_col = "file"  # will parse from filename below

        for row in reader:
            raw_patient = row.get(patient_col, "") if patient_col else ""
            # if patient_col is 'file', parse patient token from filename pattern dist_scan_patient_cropped
            if patient_col == "file" or patient_col == "File":
                # try to parse <dist>_<scan>_<patient>_cropped...
                fname = os.path.splitext(os.path.basename(raw_patient))[0]
                parts = fname.split("_")
                if len(parts) >= 3:
                    # everything between 3rd to last token (if extra) is patient, but typical is parts[2]
                    patient_token = "_".join(parts[2:-1]) if len(parts) > 4 else parts[2]
                    raw_patient = patient_token
                else:
                    # fallback to full filename
                    raw_patient = fname

            normalized = normalize_name(raw_patient)

            cobb_val = None
            reason = None

            # Exact normalized match
            if normalized in norm_cobb_map:
                cobb_val = norm_cobb_map[normalized][1]
                reason = "exact_norm"
                map_log["mapped"].append({"file": row.get("file", ""), "patient_raw": raw_patient, "patient_norm": normalized, "cobb": cobb_val})
            else:
                # Fuzzy match on normalized names
                # using difflib to find closest normalized name
                matches = difflib.get_close_matches(normalized, norm_names, n=1, cutoff=fuzzy_cutoff)
                if matches:
                    matched_norm = matches[0]
                    cobb_val = norm_cobb_map[matched_norm][1]
                    reason = f"fuzzy_norm_match({matched_norm})"
                    map_log["fuzzy_mapped"].append({"file": row.get("file", ""), "patient_raw": raw_patient, "patient_norm": normalized, "matched_norm": matched_norm, "cobb": cobb_val})
                else:
                    # no match
                    reason = "unmatched"
                    map_log["unmatched"].append({"file": row.get("file", ""), "patient_raw": raw_patient, "patient_norm": normalized})
                    unmatched_rows.append({**row, "patient": raw_patient, "patient_norm": normalized})

            # add cobb to row
            row["cobb_angle"] = cobb_val
            # ensure patient column is present and cleaned
            row["patient"] = raw_patient
            mapped_rows.append(row)

    # write mapped CSV
    with open(out_csv, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for r in mapped_rows:
            writer.writerow(r)

    # write unmatched rows for manual QA
    if unmatched_rows:
        with open(unmatched_csv, "w", newline="", encoding="utf-8") as fh:
            # write same columns plus patient_norm and patient
            cols = list(unmatched_rows[0].keys())
            writer = csv.DictWriter(fh, fieldnames=cols)
            writer.writeheader()
            for r in unmatched_rows:
                writer.writerow(r)

    # write mapping log
    with open(log_json, "w", encoding="utf-8") as jf:
        json.dump(map_log, jf, indent=2)

    print(f"Done. Wrote mapped CSV: {out_csv}")
    print(f"Unmatched rows: {len(unmatched_rows)} -> {unmatched_csv}")
    print(f"Mapping log: {log_json}")
    if collisions:
        print("Warning: collisions in normalized names found:", collisions)


# ---------- run ----------
if __name__ == "__main__":
    map_cobb_to_csv()


Done. Wrote mapped CSV: D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\all_cropped_features_with_cobb.csv
Unmatched rows: 352 -> D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\all_cropped_features_cobb_unmatched.csv
Mapping log: D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\all_cropped_features_cobb_map_log.json


In [74]:
sorted(list(patient_to_cobb.keys()))

['Aafiya Saifi',
 'Aahil',
 'Aarav Tiwari',
 'Aditya Gahrola',
 'Amit Sharma',
 'Anamika Sourout',
 'Aradhya Sahu',
 'Ayan Saifi',
 'Bhawna sheoran(o)',
 'Divyanka',
 'Gargi jha',
 'Gracy Singh',
 'Hitesh Sharma',
 'Humera Bi',
 'Imran ALI',
 'Ishika Goher',
 'Karan Bhati',
 'Kartik Verma',
 'Khushi Gupta',
 'Maya ',
 'Mrinmoy Saha',
 'Muskaan khatun',
 'Nakul',
 'Neha .delhi',
 'Nidhi',
 'Nishika',
 'Palak',
 'Radhika Sharma',
 'Satyam pal',
 'Sayba Bano',
 'Smriti Nitya',
 'Sudhanshu Singh',
 'Surjeet Barman',
 'Talib Khan',
 'Tapsya',
 'Varsha Mishra',
 'baishakhi debnath(o)',
 'karan kumar']

In [83]:
patient_to_cobb['Muskaan khatun']

39.0

In [76]:
import re
import csv
import json
import difflib
import os

# ---------- CONFIG (reuse your paths) ----------
FEATURES_CSV = r"D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\all_cropped_features.csv"
OUTPUT_CSV = FEATURES_CSV.replace(".csv", "_with_cobb.csv")
UNMATCHED_CSV = FEATURES_CSV.replace(".csv", "_cobb_unmatched.csv")
MAPPING_LOG_JSON = FEATURES_CSV.replace(".csv", "_cobb_map_log.json")

# example cobb dictionary (replace with your full dictionary)
COBB_DICT = patient_to_cobb

# ---------- helpers ----------
def normalize_name(name):
    """Normalize names to compact lowercase alnum string after removing s1..s4 or leading digits/underscores."""
    if name is None:
        return ""
    s = name.strip()
    # remove s1..s4, s01..s04, or leading digits with separators
    s = re.sub(r'^(s[1-4]|s0?[1-4]|\d+)[_\-\. ]*', '', s, flags=re.IGNORECASE)
    # remove non-alphanumeric
    s = re.sub(r'[^0-9A-Za-z ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s.lower().replace(' ', '')  # compacted

def normalize_for_tokens(name):
    """Return tokens (lowercase) from original name (keeps spaces/underscores separation)."""
    if name is None:
        return []
    s = name.strip()
    s = re.sub(r'^(s[1-4]|s0?[1-4]|\d+)[_\-\. ]*', '', s, flags=re.IGNORECASE)
    # replace non-letter/digit with space, split
    s = re.sub(r'[^0-9A-Za-z]', ' ', s)
    tokens = [t.lower() for t in s.split() if t]
    return tokens

def build_normalized_dict(cobb_dict):
    """
    Returns:
      - norm_map: normalized_name -> (orig_name, cobb)
      - firstname_map: normalized_firstname -> list of normalized_name keys
    """
    norm_map = {}
    firstname_map = {}
    collisions = {}
    for orig, val in cobb_dict.items():
        norm = normalize_name(orig)
        if norm in norm_map:
            collisions.setdefault(norm, []).append((orig, val))
            continue
        norm_map[norm] = (orig, val)
        # build firstname token (from original orig string)
        tokens = normalize_for_tokens(orig)
        first = tokens[0] if tokens else norm
        first_norm = re.sub(r'[^0-9A-Za-z]', '', first).lower()
        firstname_map.setdefault(first_norm, []).append(norm)
    return norm_map, firstname_map, collisions

# ---------- main mapping with first-name fallback ----------
def map_cobb_to_csv(features_csv=FEATURES_CSV, out_csv=OUTPUT_CSV, unmatched_csv=UNMATCHED_CSV,
                    cobb_dict=COBB_DICT, log_json=MAPPING_LOG_JSON, fuzzy_cutoff=0.8):
    if not os.path.exists(features_csv):
        raise FileNotFoundError(f"Features CSV not found: {features_csv}")

    norm_map, firstname_map, collisions = build_normalized_dict(cobb_dict)
    norm_names = list(norm_map.keys())

    mapped_rows = []
    unmatched_rows = []
    map_log = {"mapped": [], "firstname_mapped": [], "fuzzy_mapped": [], "unmatched": [], "collisions": collisions}

    with open(features_csv, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        fieldnames = list(reader.fieldnames) if reader.fieldnames else []
        if "cobb_angle" not in fieldnames:
            fieldnames.append("cobb_angle")
        # find patient column if present
        patient_col_candidates = ["patient", "Patient", "patient_name", "name", "file"]
        patient_col = None
        for cand in patient_col_candidates:
            if cand in fieldnames:
                patient_col = cand
                break

        for row in reader:
            raw_patient = row.get(patient_col, "") if patient_col else ""
            # if patient_col is file, parse token from filename
            if patient_col == "file":
                fname = os.path.splitext(os.path.basename(raw_patient))[0]
                parts = fname.split("_")
                if len(parts) >= 3:
                    patient_token = "_".join(parts[2:-1]) if len(parts) > 4 else parts[2]
                    raw_patient = patient_token

            # Create normalized forms
            compact_norm = normalize_name(raw_patient)  # compacted
            tokens = normalize_for_tokens(raw_patient)  # token list (first name likely tokens[0])
            first_token = tokens[0] if tokens else compact_norm

            cobb_val = None
            reason = None

            # 1) Exact normalized match
            if compact_norm and compact_norm in norm_map:
                cobb_val = norm_map[compact_norm][1]
                reason = "exact_norm"
                map_log["mapped"].append({"file": row.get("file", ""), "patient_raw": raw_patient, "patient_norm": compact_norm, "cobb": cobb_val})

            else:
                # 2) First-name only matching (if CSV patient looks like a single token OR tokens length ==1)
                #    Identify candidates in firstname_map by first_token; also consider substring matches.
                candidates = []
                first_norm = re.sub(r'[^0-9A-Za-z]', '', first_token).lower()
                if first_norm in firstname_map:
                    candidates = firstname_map[first_norm].copy()

                # also consider norm_map keys that contain the first token as substring (for merged names)
                if not candidates and first_norm:
                    for nm in norm_names:
                        if first_norm in nm:
                            candidates.append(nm)

                # If exactly one candidate, map it
                if len(candidates) == 1:
                    matched_norm = candidates[0]
                    cobb_val = norm_map[matched_norm][1]
                    reason = f"firstname_unique({matched_norm})"
                    map_log["firstname_mapped"].append({"file": row.get("file", ""), "patient_raw": raw_patient, "patient_norm": compact_norm, "matched_norm": matched_norm, "cobb": cobb_val})
                else:
                    # 3) Fuzzy match on normalized names
                    matches = difflib.get_close_matches(compact_norm, norm_names, n=1, cutoff=fuzzy_cutoff)
                    if matches:
                        matched_norm = matches[0]
                        cobb_val = norm_map[matched_norm][1]
                        reason = f"fuzzy_norm_match({matched_norm})"
                        map_log["fuzzy_mapped"].append({"file": row.get("file", ""), "patient_raw": raw_patient, "patient_norm": compact_norm, "matched_norm": matched_norm, "cobb": cobb_val})
                    else:
                        # unmatched
                        reason = "unmatched"
                        map_log["unmatched"].append({"file": row.get("file", ""), "patient_raw": raw_patient, "patient_norm": compact_norm})
                        unmatched_rows.append({**row, "patient": raw_patient, "patient_norm": compact_norm})

            row["cobb_angle"] = cobb_val
            row["patient"] = raw_patient
            mapped_rows.append(row)

    # write mapped CSV
    with open(out_csv, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for r in mapped_rows:
            writer.writerow(r)

    # write unmatched rows for QA
    if unmatched_rows:
        with open(unmatched_csv, "w", newline="", encoding="utf-8") as fh:
            cols = list(unmatched_rows[0].keys())
            writer = csv.DictWriter(fh, fieldnames=cols)
            writer.writeheader()
            for r in unmatched_rows:
                writer.writerow(r)

    # write mapping log
    with open(log_json, "w", encoding="utf-8") as jf:
        json.dump(map_log, jf, indent=2)

    print(f"Done. Wrote mapped CSV: {out_csv}")
    print(f"Unmatched rows: {len(unmatched_rows)} -> {unmatched_csv}")
    print(f"Mapping log: {log_json}")
    if collisions:
        print("Warning: collisions in normalized names found:", collisions)


# Run
if __name__ == "__main__":
    map_cobb_to_csv()


Done. Wrote mapped CSV: D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\all_cropped_features_with_cobb.csv
Unmatched rows: 172 -> D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\all_cropped_features_cobb_unmatched.csv
Mapping log: D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\all_cropped_features_cobb_map_log.json


In [90]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------- CONFIG ----------
DATA_PATH = r"D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\all_cropped_features_with_cobb.csv"
TEST_PATIENTS = ["gargi", "bhavna"]   # change as needed
TARGET = "cobb_angle"

# ---------- Load & Clean ----------
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)

# Keep only valid rows
df = df.dropna(subset=[TARGET])
df = df[df[TARGET] > 0]

# Normalize patient names (for grouping)
df["patient_norm"] = (
    df["patient"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]", "", regex=True)
)

# ---------- Aggregate to one row per patient ----------
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
group_cols = ["patient_norm", "patient"]
agg_df = df.groupby(group_cols)[numeric_cols].median().reset_index()

print(f"\nAggregated to {len(agg_df)} patients")

# ---------- Split Train/Test ----------
test_patients_norm = [p.lower().replace(" ", "").replace("_", "") for p in TEST_PATIENTS]

train_df = agg_df[~agg_df["patient_norm"].isin(test_patients_norm)].copy()
test_df  = agg_df[agg_df["patient_norm"].isin(test_patients_norm)].copy()

print(f"Training on {len(train_df)} patients, Testing on {len(test_df)} patients")

# ---------- Features ----------
non_features = ["patient", "patient_norm", TARGET]
features = [c for c in agg_df.columns if c not in non_features and pd.api.types.is_numeric_dtype(agg_df[c])]

X_train = train_df[features].fillna(0)
y_train = train_df[TARGET]
X_test  = test_df[features].fillna(0)
y_test  = test_df[TARGET]

print("Feature count:", len(features))

# ---------- Scale & Train ----------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVR(kernel="rbf", C=10, epsilon=1.0, gamma="scale")
svm.fit(X_train_scaled, y_train)

# ---------- Evaluate ----------
# ---------- Evaluate ----------
y_pred = svm.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n📊 Evaluation Results (Patient-Level)")
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

# ---------- Detailed Results ----------
results = pd.DataFrame({
    "patient": test_df["patient"],
    "true_cobb": y_test,
    "pred_cobb": y_pred
})
print("\nPatient-level predictions:")
print(results)

# Optional save
out_path = os.path.join(os.path.dirname(DATA_PATH), "patient_level_results.csv")
results.to_csv(out_path, index=False)
print("\nSaved patient-level results to:", out_path)


Loaded: (509, 22)

Aggregated to 17 patients
Training on 15 patients, Testing on 2 patients
Feature count: 17

📊 Evaluation Results (Patient-Level)
MAE  : 3.293
RMSE : 3.539
R²   : -0.391

Patient-level predictions:
  patient  true_cobb  pred_cobb
2  bhavna       54.0  58.589083
4   gargi       60.0  58.003479

Saved patient-level results to: D:\Professional\Projects\Aiims\Gait\Data\project work\Surface topography data\All_cropped\patient_level_results.csv


In [91]:
# ---------- Save Model ----------
import joblib

model_path =  "svm_cobb_model.pkl"

joblib.dump(
    {
        "model": svm,
        "scaler": scaler,
        "features": features
    },
    model_path
)

print(f"\n💾 Model saved successfully at: {model_path}")



💾 Model saved successfully at: svm_cobb_model.pkl
